In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_22.pickle

In [ ]:
%%cudf.pandas.profile
### cell 22 ###
def get_n_grams(n_grams, top_n=10):
    dfs = []
    for dt in tqdm(train.discourse_type.unique()):
        # GPU‐accelerated boolean filtering
        df_dt = train[train.discourse_type == dt]
        texts = df_dt.discourse_text.tolist()

        # CountVectorizer still runs on CPU under the extension
        vec = CountVectorizer(
            lowercase=True,
            stop_words='english',
            ngram_range=(n_grams, n_grams)
        ).fit(texts)
        bag_of_words = vec.transform(texts)

        # aggregate counts
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx])
                      for word, idx in vec.vocabulary_.items()]

        # build DataFrame exactly as original to preserve its index
        cvec_df = pd.DataFrame.from_records(
            words_freq,
            columns=['words', 'counts']
        ).sort_values(by='counts', ascending=False)
        cvec_df.insert(0, 'Discourse_type', dt)
        cvec_df = cvec_df.iloc[:top_n, :]

        dfs.append(cvec_df)

    # concatenate without resetting the index to match the original DataFrame's index
    return pd.concat(dfs)

bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()